## **U-Net con Kvasir-SEG**

En este fichero se procede a evaluar el rendimiento de una arquitectura clásica como lo es la U-Net en la segmentación de pólipos usando el dataset de Kvasir-SEG.

En este primer bloque de código se importan las librerías necesarias para la ejecución del fichero completo. Además, se importan las funciones de métricas necesarias, para las cuales se tuvo que añadir la raíz del proyecto al path de Python para que los imports funcionasen correctamente. Finalmente, se define la ruta al dataset con el que entrenaremos los modelos.

In [3]:
from torch.utils.data import Dataset, DataLoader
from torch.optim import Adam
from torch.optim.lr_scheduler import StepLR
import segmentation_models_pytorch as smp
import numpy as np
import os
import cv2
import pandas as pd
import time
import sys
import torch
import torch.nn as nn


root_path = os.path.abspath('..')
if root_path not in sys.path:
    sys.path.append(root_path)

from utils.segmentation_quality_metrics import boundary_iou, hausdorff_95, compute_all_metrics

dataset = "C:\\Users\\DanielTalavera\\Desktop\\Trabajo_Fin_de_Grado\\Kvasir-SEG"


En el siguiente bloque de código, se define el dataset cargando en el constructor las rutas de las imágenes y máscaras del split correspondiente (train, val o test). Se filtran aquellas muestras cuya máscara no existe o está vacía, y se generan los pares imagen-máscara válidos.

En el método __getitem__ se aplica el preprocesamiento de cada muestra. Tanto la imagen como la máscara se redimensionan a 512x512. La imagen se normaliza entre 0 y 1 y se convierte a tensor. Además, la máscara se binariza con umbral 127 y también se convierte a tensor, en este caso con una dimensión de canal adicional para que sea compatible con la salida de U-Net, que tiene forma (1, H, W). 

In [4]:
class KvasirDataset(Dataset):
    def __init__(self, dataset_path, split, img_size=512):
        self.img_size = img_size
        self.images_dir = os.path.join(dataset_path, "images", split)
        self.masks_dir = os.path.join(dataset_path, "masks", split)
        self.samples = []

        for img_file in os.listdir(self.images_dir):
            if not img_file.endswith(".jpg"):
                continue
            name = os.path.splitext(img_file)[0]
            img_path = os.path.join(self.images_dir, img_file)
            mask_path = os.path.join(self.masks_dir, name + ".jpg")
            if not os.path.exists(mask_path):
                continue
            gt = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
            if gt is None or (gt > 127).sum() == 0:
                continue
            self.samples.append((img_path, mask_path))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, mask_path = self.samples[idx]

        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        image = cv2.resize(image, (self.img_size, self.img_size))
        image = torch.tensor(image).permute(2, 0, 1).float() / 255.0

        mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        mask = cv2.resize(mask, (self.img_size, self.img_size))
        mask = torch.tensor((mask > 127).astype(np.float32)).unsqueeze(0)

        return image, mask


En primer lugar, se habilita TF32 para las operaciones matriciales y se define el dispositivo donde se ejecutará el entrenamiento.

La función train_unet instancia una U-Net con encoder ResNet34 preentrenado en ImageNet, lo que permite aprovechar características visuales generales aprendidas en un dominio amplio antes de especializarse en pólipos. El modelo tiene tres canales de entrada (RGB) y un canal de salida (segmentación binaria). Como función de pérdida se utiliza BCEWithLogitsLoss, el optimizador es Adam con tasa de aprendizaje inicial de 1e-4 y se utiliza un scheduler para reducir la tasa de aprendizaje a la mitad cada 20 épocas para afinar el entrenamiento en las últimas iteraciones.

In [5]:
torch.backends.cuda.matmul.allow_tf32 = True
device = torch.device("cuda:0")
assert torch.cuda.is_available(), "CUDA no disponible"

def train_unet(dataset_path, output_weights, epochs=50, batch_size=4):
    model = smp.Unet(
        encoder_name="resnet34",
        encoder_weights="imagenet",
        in_channels=3,
        classes=1
    )
    model.to(device)

    optimizer = Adam(model.parameters(), lr=1e-4)
    scheduler = StepLR(optimizer, step_size=20, gamma=0.5)
    loss_fn = nn.BCEWithLogitsLoss()

    train_dataset = KvasirDataset(dataset_path, "train")
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

    model.train()
    for epoch in range(epochs):
        total_loss = 0
        for images, masks in train_loader:
            images, masks = images.to(device), masks.to(device)
            preds = model(images)
            loss = loss_fn(preds, masks)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        scheduler.step()
        print(f"Epoch {epoch+1}/{epochs} - Loss: {total_loss/len(train_loader):.4f}")

    torch.save(model.state_dict(), output_weights)
    return output_weights


La función evaluate_unet carga los pesos entrenados sobre una nueva instancia de U-Net con encoder_weights=None, ya que en evaluación no se necesita el preentrenamiento de ImageNet sino los pesos específicos del dominio. Para cada muestra del conjunto de test, la imagen se redimensiona a 512x512 para la inferencia y la máscara predicha se redimensiona de vuelta a las dimensiones originales para poder compararla con el ground truth. La salida del modelo se pasa por una sigmoide y se binariza con umbral 0.5 para obtener la máscara final. En cada inferencia se miden la latencia y el consumo de VRAM, y se calculan todas las métricas. Al final se devuelve un diccionario con la media de cada métrica sobre el conjunto de test.

Por último, se lanzan primero el entrenamiento y luego la evaluación, midiendo el tiempo que tarda cada uno. Ambos tiempos se añaden al diccionario de resultados, que finalmente se guardan en un CSV. Si el fichero ya existe se añade una nueva línea sin sobreescribir los resultados anteriores para poder comparar los resultados de distintos experimentos. 

In [6]:
def evaluate_unet(dataset_path, weights_path):
    model = smp.Unet(
        encoder_name="resnet34",
        encoder_weights=None,
        in_channels=3,
        classes=1
    )
    model.load_state_dict(torch.load(weights_path))
    model.to(device)
    model.eval()

    test_dataset = KvasirDataset(dataset_path, "test")

    iou_scores, precision_scores, recall_scores, f1_scores = [], [], [], []
    dice_scores, specificity_scores, f2_scores, pixel_accuracy_scores = [], [], [], []
    boundary_iou_scores, hausdorff_95_scores, latency_scores, vram_scores = [], [], [], []

    for img_path, mask_path in test_dataset.samples:
        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        H, W = image.shape[:2]

        gt_mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        gt_mask = (gt_mask > 127).astype(bool)

        image_resized = cv2.resize(image, (512, 512))
        image_tensor = torch.tensor(image_resized).permute(2, 0, 1).float() / 255.0
        image_tensor = image_tensor.unsqueeze(0).to(device)

        if torch.cuda.is_available():
            vram_before = torch.cuda.memory_allocated() / 1024**2

        start = time.time()
        with torch.no_grad():
            pred = model(image_tensor)
        latency = (time.time() - start) * 1000

        if torch.cuda.is_available():
            vram = torch.cuda.memory_allocated() / 1024**2 - vram_before
        else:
            vram = 0

        pred_mask = torch.sigmoid(pred).squeeze().cpu().numpy()
        pred_mask = (pred_mask > 0.5).astype(np.uint8)
        pred_mask = cv2.resize(pred_mask, (W, H), interpolation=cv2.INTER_NEAREST).astype(bool)

        iou, precision, recall, f1, dice, specificity, f2, pixel_accuracy = compute_all_metrics(pred_mask, gt_mask)
        iou_scores.append(iou)
        precision_scores.append(precision)
        recall_scores.append(recall)
        f1_scores.append(f1)
        dice_scores.append(dice)
        specificity_scores.append(specificity)
        f2_scores.append(f2)
        pixel_accuracy_scores.append(pixel_accuracy)
        boundary_iou_scores.append(boundary_iou(pred_mask, gt_mask))
        hausdorff_95_scores.append(hausdorff_95(pred_mask, gt_mask))
        latency_scores.append(latency)
        vram_scores.append(vram)

    results = {
        "modelo": ["unet_resnet34_kvasir"],
        "mean_iou": [np.mean(iou_scores)],
        "mean_f1": [np.mean(f1_scores)],
        "mean_recall": [np.mean(recall_scores)],
        "mean_precision": [np.mean(precision_scores)],
        "mean_dice": [np.mean(dice_scores)],
        "mean_specificity": [np.mean(specificity_scores)],
        "mean_f2": [np.mean(f2_scores)],
        "mean_pixel_accuracy": [np.mean(pixel_accuracy_scores)],
        "mean_boundary_iou": [np.mean(boundary_iou_scores)],
        "mean_hausdorff_95": [np.mean(hausdorff_95_scores)],
        "mean_latency_ms": [np.mean(latency_scores)],
        "mean_vram_mb": [np.mean(vram_scores)]
    }

    return results

output_weights = "C:\\Users\\DanielTalavera\\Desktop\\Trabajo_Fin_de_Grado\\pesos_finetuned\\pesos_unet_resnet34_kvasir.pt"
output_path = "C:\\Users\\DanielTalavera\\Desktop\\Trabajo_Fin_de_Grado\\resultados_fine_tunning\\resultados_unet.csv"

start_train = time.time()
trained_weights = train_unet(dataset, output_weights)
train_time = time.time() - start_train

start_eval = time.time()
results = evaluate_unet(dataset, trained_weights)
eval_time = time.time() - start_eval

results["train_time_minutes"] = [round(train_time / 60, 2)]
results["eval_time_minutes"] = [round(eval_time / 60, 2)]

df = pd.DataFrame(results)
if os.path.exists(output_path):
    df.to_csv(output_path, mode='a', header=False, index=False)
else:
    df.to_csv(output_path, index=False)

c:\Users\DanielTalavera\AppData\Local\Programs\Python\Python310\lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\DanielTalavera\.cache\huggingface\hub\models--smp-hub--resnet34.imagenet. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


Epoch 1/50 - Loss: 0.4396
Epoch 2/50 - Loss: 0.2481
Epoch 3/50 - Loss: 0.1693
Epoch 4/50 - Loss: 0.1391
Epoch 5/50 - Loss: 0.0967
Epoch 6/50 - Loss: 0.0914
Epoch 7/50 - Loss: 0.0735
Epoch 8/50 - Loss: 0.0638
Epoch 9/50 - Loss: 0.0532
Epoch 10/50 - Loss: 0.0497
Epoch 11/50 - Loss: 0.0499
Epoch 12/50 - Loss: 0.0355
Epoch 13/50 - Loss: 0.0276
Epoch 14/50 - Loss: 0.0242
Epoch 15/50 - Loss: 0.0232
Epoch 16/50 - Loss: 0.0409
Epoch 17/50 - Loss: 0.0355
Epoch 18/50 - Loss: 0.0281
Epoch 19/50 - Loss: 0.0227
Epoch 20/50 - Loss: 0.0213
Epoch 21/50 - Loss: 0.0190
Epoch 22/50 - Loss: 0.0162
Epoch 23/50 - Loss: 0.0151
Epoch 24/50 - Loss: 0.0143
Epoch 25/50 - Loss: 0.0135
Epoch 26/50 - Loss: 0.0132
Epoch 27/50 - Loss: 0.0126
Epoch 28/50 - Loss: 0.0122
Epoch 29/50 - Loss: 0.0119
Epoch 30/50 - Loss: 0.0114
Epoch 31/50 - Loss: 0.0108
Epoch 32/50 - Loss: 0.0108
Epoch 33/50 - Loss: 0.0103
Epoch 34/50 - Loss: 0.0099
Epoch 35/50 - Loss: 0.0098
Epoch 36/50 - Loss: 0.0096
Epoch 37/50 - Loss: 0.0091
Epoch 38/5

C:\Users\DanielTalavera\AppData\Local\Temp\ipykernel_7596\4035751209.py:8: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(weights_path))
